<a href="https://colab.research.google.com/github/balla-a12/quant-equity-research/blob/main/05_offexchange_signal_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Off-Exchange Signal

The fourth and final input, and the one shaped differently from the rest. The congress,
contracts, and lobbying signals read discrete *events*; this one reads trading *flow* —
off-exchange (dark-pool) volume, where large and often institutional participants trade
away from the lit market.

### A confirming signal, by design

Off-exchange flow is genuinely useful but genuinely ambiguous. A high and rising
off-exchange share shows a heavier institutional footprint in a name, yet the raw volume
does not reveal direction — the same flow can be accumulation or distribution. So this
signal reads institutional *engagement* rather than making a directional call, carries
the **lowest weight** in the composite, and earns its keep by corroborating a read the
other three signals already make.

| Feature | What it captures |
|---|---|
| `dpi` | Off-exchange short ratio (`OTC_Short / OTC_Total`) — a buying-pressure proxy |
| `volume` | Off-exchange volume (`OTC_Total`) — institutional footprint |

### What the live feed actually gives us

The off-exchange endpoint returns a **daily snapshot**, one row per ticker, with columns
`OTC_Short`, `OTC_Total`, and `DPI` (which equals `OTC_Short / OTC_Total`). Two facts
shape the design:

- **No total-market-volume column**, so "off-exchange share of all volume" is not
  computable here — the footprint feature uses the off-exchange volume level directly.
- **No per-ticker history** (one date), so there is no trend feature, and this signal
  cannot be backtested historically from the live feed. It is a current-state confirming
  overlay. The mock spans several dates only so the backtest engine has something to step
  through.

The feed covers the entire market (~$5{,}400$ tickers), most of them illiquid, so a
**liquidity floor** (`min_oe_volume`) drops thinly-traded names whose ratios are
degenerate — a micro-cap can print `DPI = 1.0` on a single sparse trade.


## Setup

Defaults to mock data; set `USE_LIVE = True` for live Quiver off-exchange data.


In [2]:
!pip install -q quiverquant pandas numpy sqlalchemy plotly pyyaml requests

import subprocess, os, sys
from google.colab import userdata

GITHUB_USER = "balla-a12"   # <-- edit this
REPO = "quant-equity-research"

gh_token = userdata.get("GITHUB_TOKEN")
url = f"https://{gh_token}@github.com/{GITHUB_USER}/{REPO}.git"
r = subprocess.run(["git", "clone", url], capture_output=True, text=True)
if r.returncode == 0 or "already exists" in r.stderr:
    os.chdir(REPO)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], capture_output=True, text=True)
print("Working in:", os.getcwd())

try:
    QUIVER_TOKEN = userdata.get("QUIVER_API_KEY")
except Exception:
    QUIVER_TOKEN = None
USE_LIVE = True   # flip to True for live Quiver data

Working in: /content/quant-equity-research


## Write the signal module

The signal and the off-exchange ingestion (a new `off_exchange()` method on the client,
plus a tolerant normalizer and mock feed) go into the package.


In [3]:
open("src/quant_research/signals/offexchange.py", "w").write('"""Off-exchange (dark-pool) activity signal.\n\nA *confirming* signal, not a directional one. The Quiver off-exchange feed is a daily\nsnapshot, one row per ticker, reporting off-exchange volume (``OTC_Total``), the short\nportion of it (``OTC_Short``), and ``DPI = OTC_Short / OTC_Total`` -- the dark-pool short\nratio. Under the common reading of this metric, off-exchange short volume is the\nmarket-maker side of lit buying (you buy, the dark-pool MM sells short to fill you), so a\nhigher ratio is interpreted as net accumulation. That interpretation is contested, which\nis why this signal carries the lowest weight in the composite and is used to corroborate\nrather than to call direction on its own.\n\nBecause the feed is a single snapshot with no per-ticker history, the signal is purely\ncross-sectional -- there is no trend feature, and it cannot be backtested historically\nfrom the live feed (the mock spans several dates so the engine still has something to\nstep through).\n\nFeatures (each normalized across the cross-section, then weighted):\n  dpi     - off-exchange short ratio; buying-pressure proxy\n  volume  - off-exchange volume (OTC_Total); institutional footprint\n\nA liquidity floor (``min_oe_volume``) drops thinly-traded names whose ratios are\ndegenerate (e.g. a micro-cap that prints DPI = 1.0 on a single sparse trade).\n"""\nfrom datetime import date\nimport pandas as pd\n\nfrom .base import BaseSignal\n\nDEFAULT_WEIGHTS = {\n    "dpi": 0.65,\n    "volume": 0.35,\n}\n\n\nclass OffExchangeSignal(BaseSignal):\n    name = "off_exchange"\n    description = "Off-exchange (dark-pool) buying-pressure proxy; a confirming signal."\n\n    def __init__(self, client, lookback_days=60, min_oe_volume=1_000_000, weights=None):\n        self.client = client\n        self.lookback_days = lookback_days\n        self.min_oe_volume = min_oe_volume\n        self.weights = weights or dict(DEFAULT_WEIGHTS)\n\n    def compute(self, as_of=None, flow=None):\n        as_of = pd.Timestamp(as_of or date.today())\n        start = as_of - pd.Timedelta(days=self.lookback_days)\n\n        df = self.client.off_exchange() if flow is None else flow\n        win = df[(df.date > start) & (df.date <= as_of)].copy()\n        if win.empty:\n            return pd.DataFrame(columns=["score"])\n\n        # one row per ticker (the feed is a daily snapshot); take the most recent\n        win = win.sort_values("date").groupby("ticker", as_index=False).tail(1)\n        win = win[win.oe_volume >= self.min_oe_volume]\n        if win.empty:\n            return pd.DataFrame(columns=["score"])\n\n        feat = win.set_index("ticker")[["dpi", "oe_volume"]].rename(\n            columns={"oe_volume": "volume"})\n        norm = pd.DataFrame({f: self._norm(feat[f]) for f in self.weights})\n        raw = sum(self.weights[f] * norm[f] for f in self.weights)\n        feat["score"] = self._scale_0_100(raw).round(1)\n        for f in norm.columns:\n            feat[f + "_n"] = norm[f].round(3)\n        return feat.sort_values("score", ascending=False)\n\n    @staticmethod\n    def _norm(s):\n        s = s.astype(float)\n        spread = s.max() - s.min()\n        return (s - s.min()) / spread if spread > 0 else s * 0.0\n')
open("src/quant_research/ingestion/client.py", "w").write('"""A unified client for Quiver Quantitative data.\n\nThe QuiverClient exposes one method per dataset. Each method fetches raw data\n(from the live API when a token is supplied, otherwise from the mock generator)\nand returns it normalized into a consistent internal schema. The rest of the\nproject depends only on that internal schema, so nothing downstream needs to\nknow or care whether the data came from the live API or the mock source.\n"""\nimport re\nimport pandas as pd\n\nfrom . import mock_data\n\n\ndef _parse_range(text):\n    """Turn a Quiver amount range like \'$15,001 - $50,000\' into (low, high)."""\n    nums = re.findall(r"[\\d,]+", str(text))\n    vals = [int(n.replace(",", "")) for n in nums if n.replace(",", "").isdigit()]\n    if len(vals) >= 2:\n        return vals[0], vals[1]\n    if len(vals) == 1:\n        return vals[0], vals[0]\n    return 0, 0\n\n\nclass QuiverClient:\n    def __init__(self, token=None, mock=False, mock_history_days=40):\n        self.mock_history_days = mock_history_days\n        # No token means we cannot reach the live API, so we fall back to mock.\n        self.mock = mock or token is None\n        self._api = None\n        if not self.mock:\n            import quiverquant            # imported lazily; unused in mock mode\n            self._api = quiverquant.quiver(token)\n\n    # ---- Congressional trades -------------------------------------------\n    def congress_trades(self, historical=False):\n        if self.mock:\n            raw = mock_data.mock_congress_trading(\n                history_days=self.mock_history_days,\n                n=max(180, self.mock_history_days * 4))\n        else:\n            raw = self._api.congress_trading(recent=not historical)\n        return self._normalize_congress(raw)\n\n    @staticmethod\n    def _col(df, *names, default=""):\n        """First present column among `names`, else a default-filled Series.\n\n        The recent and bulk Quiver endpoints differ in their column names, so\n        every field is looked up tolerantly rather than assumed.\n        """\n        for n in names:\n            if n in df.columns:\n                return df[n]\n        return pd.Series([default] * len(df), index=df.index)\n\n    def _normalize_congress(self, df):\n        df = df.copy()\n        if "Range" in df.columns:\n            lows, highs = zip(*df["Range"].map(_parse_range)) if len(df) else ([], [])\n            amount_min, amount_max = list(lows), list(highs)\n        else:\n            amt = (self._col(df, "Amount", "Trade_Size_USD", "amount", default=0)\n                   .astype(str).str.replace(r"[$,]", "", regex=True))\n            amt = pd.to_numeric(amt, errors="coerce").fillna(0)\n            amount_min, amount_max = amt.tolist(), amt.tolist()\n        out = pd.DataFrame({\n            "ticker": self._col(df, "Ticker").astype(str).str.upper(),\n            "transaction_date": pd.to_datetime(self._col(df, "TransactionDate", "Traded"),\n                                               errors="coerce"),\n            "report_date": pd.to_datetime(self._col(df, "ReportDate", "Filed"),\n                                          errors="coerce"),\n            "representative": self._col(df, "Representative", "Name"),\n            "party": self._col(df, "Party"),\n            "chamber": self._col(df, "House", "Chamber"),\n            "transaction_type": self._col(df, "Transaction").astype(str).str.strip().str.title(),\n            "amount_min": amount_min,\n            "amount_max": amount_max,\n        })\n        return out\n\n    # ---- Insider transactions -------------------------------------------\n    def insider_trades(self):\n        raw = (mock_data.mock_insiders() if self.mock\n               else self._api.insiders())\n        return self._normalize_insiders(raw)\n\n    def _normalize_insiders(self, df):\n        df = df.copy()\n        code_map = {"P": "Purchase", "S": "Sale"}\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "transaction_date": pd.to_datetime(df["Date"]),\n            "insider_name": df["Name"],\n            "title": df["Title"],\n            "transaction_type": df["TransactionCode"].map(code_map).fillna("Other"),\n            "shares": df["Shares"].astype(int),\n            "price_per_share": df["PricePerShare"].astype(float),\n        })\n        out["value"] = (out["shares"] * out["price_per_share"]).round(2)\n        return out\n\n    # ---- Government contracts --------------------------------------------\n    def gov_contracts(self):\n        if self.mock:\n            days = max(self.mock_history_days, 120)\n            raw = mock_data.mock_gov_contracts(history_days=days, n=max(80, days * 2))\n        else:\n            raw = self._api.gov_contracts()\n        return self._normalize_gov(raw)\n\n    def _normalize_gov(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "award_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "agency": df["Agency"],\n        })\n        return out\n\n    # ---- Lobbying --------------------------------------------------------\n    def lobbying(self):\n        if self.mock:\n            days = max(self.mock_history_days, 360)\n            raw = mock_data.mock_lobbying(history_days=days, n=max(140, days))\n        else:\n            raw = self._api.lobbying()\n        return self._normalize_lobbying(raw)\n\n    def _normalize_lobbying(self, df):\n        df = df.copy()\n        out = pd.DataFrame({\n            "ticker": df["Ticker"].str.upper(),\n            "filing_date": pd.to_datetime(df["Date"]),\n            "amount": df["Amount"].astype(float),\n            "client": df["Client"],\n            "issue": df["Issue"],\n        })\n        return out\n\n    # ---- Off-exchange (dark pool) volume ---------------------------------\n    def off_exchange(self):\n        if self.mock:\n            days = max(self.mock_history_days, 90)\n            raw = mock_data.mock_offexchange(history_days=days)\n        else:\n            raw = self._api.offexchange()\n        return self._normalize_offexchange(raw)\n\n    def _normalize_offexchange(self, df):\n        df = df.copy()\n        oe_vol = pd.to_numeric(self._col(df, "OTC_Total", "OffExchange", default=0), errors="coerce")\n        oe_short = pd.to_numeric(self._col(df, "OTC_Short", "Short", default=0), errors="coerce")\n        out = pd.DataFrame({\n            "ticker": self._col(df, "Ticker").astype(str).str.upper(),\n            "date": pd.to_datetime(self._col(df, "Date"), errors="coerce"),\n            "oe_volume": oe_vol,\n            "oe_short": oe_short,\n        })\n        if "DPI" in df.columns:\n            out["dpi"] = pd.to_numeric(df["DPI"], errors="coerce")\n        else:                                            # DPI = off-exchange short ratio\n            out["dpi"] = (out.oe_short / out.oe_volume.replace(0, pd.NA)).clip(0, 1)\n        return out\n')
open("src/quant_research/ingestion/mock_data.py", "w").write('"""Synthetic data shaped like the Quiver Quantitative API responses.\n\nColumn names and value formats mirror the live `quiverquant` package so the same\nnormalization works on mock and real data. A few tickers carry heavy, purchase-\nskewed activity, and some buys are routed to representatives whose committee aligns\nwith the traded sector, so the signal\'s features have structure to detect. Member\nnames are real, so the live enrichment layer resolves them directly.\n"""\nimport numpy as np\nimport pandas as pd\nfrom datetime import date, timedelta\n\nfrom ..enrichment import mock as menr\n\n_UNIVERSE = ["PLTR", "LMT", "RTX", "AXON", "NOC", "CELH", "MELI", "CAVA",\n             "NVDA", "AAPL", "MSFT", "GE", "BA", "CAT", "JPM"]\n_HOT = ["PLTR", "LMT", "AXON"]\n_RANGES = ["$1,001 - $15,000", "$15,001 - $50,000", "$50,001 - $100,000",\n           "$100,001 - $250,000", "$250,001 - $500,000", "$500,001 - $1,000,000"]\n_REPS = list(menr.REP_COMMITTEE.keys())\n_TITLES = ["CEO", "CFO", "Director", "President", "EVP", "VP Finance", "COO"]\n_AGENCIES = ["Department of Defense", "Department of Energy", "NASA",\n             "Department of Homeland Security", "Department of Health"]\n_ISSUES = ["Defense", "Healthcare", "Taxation", "Energy", "Technology", "Trade"]\n\n\ndef _recent(rng, max_days_ago, min_days_ago=0):\n    return date.today() - timedelta(days=int(rng.integers(min_days_ago, max_days_ago)))\n\n\ndef mock_congress_trading(seed=42, n=180, history_days=40):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        if rng.random() < 0.45:\n            ticker = rng.choice(_HOT)\n            txn = rng.choice(["Purchase", "Sale"], p=[0.85, 0.15])\n        else:\n            ticker = rng.choice(_UNIVERSE)\n            txn = rng.choice(["Purchase", "Sale"], p=[0.55, 0.45])\n\n        sector = menr.TICKER_SECTOR.get(ticker)\n        aligned = menr.SECTOR_REPS.get(sector, [])\n        rep = rng.choice(aligned) if (aligned and rng.random() < 0.5) else rng.choice(_REPS)\n\n        report = _recent(rng, history_days)\n        transaction = report - timedelta(days=int(rng.integers(10, 45)))\n        rows.append({\n            "Representative": rep,\n            "Party": "D" if _REPS.index(rep) % 2 == 0 else "R",\n            "House": rng.choice(["Representatives", "Senate"], p=[0.75, 0.25]),\n            "Ticker": ticker, "Transaction": txn, "Range": rng.choice(_RANGES),\n            "TransactionDate": transaction, "ReportDate": report,\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_insiders(seed=43, n=120):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_HOT) if rng.random() < 0.4 else rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Name": f"Insider {rng.integers(1000, 9999)}",\n            "Title": rng.choice(_TITLES), "Date": _recent(rng, 90),\n            "TransactionCode": rng.choice(["P", "S"], p=[0.6, 0.4]),\n            "Shares": int(rng.integers(500, 50000)),\n            "PricePerShare": round(float(rng.uniform(20, 400)), 2),\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_gov_contracts(seed=44, n=80, history_days=120):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_HOT) if rng.random() < 0.5 else rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Date": _recent(rng, history_days),\n            "Amount": int(rng.integers(50_000, 500_000_000)),\n            "Agency": rng.choice(_AGENCIES), "Description": "Procurement contract award",\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_lobbying(seed=45, n=140, history_days=360):\n    rng = np.random.default_rng(seed)\n    rows = []\n    for _ in range(n):\n        ticker = rng.choice(_UNIVERSE)\n        rows.append({\n            "Ticker": ticker, "Date": _recent(rng, history_days),\n            "Amount": int(rng.integers(10_000, 5_000_000)),\n            "Client": f"{ticker} Inc.", "Issue": rng.choice(_ISSUES),\n        })\n    return pd.DataFrame(rows)\n\n\ndef mock_offexchange(seed=46, history_days=90):\n    """Synthetic off-exchange data matching Quiver\'s live schema.\n\n    Columns: Ticker, Date, OTC_Short, OTC_Total, DPI (= OTC_Short / OTC_Total).\n    The live feed is a single daily snapshot per ticker; the mock spans several dates so\n    backtests have history to step through. Hot tickers carry a higher DPI (read as more\n    off-exchange buying pressure) and heavier off-exchange volume.\n    """\n    rng = np.random.default_rng(seed)\n    rows = []\n    steps = max(history_days // 3, 1)\n    for ticker in _UNIVERSE:\n        base_dpi = 0.50 + (0.07 if ticker in _HOT else 0.0) + rng.uniform(-0.02, 0.02)\n        base_vol = int(rng.integers(5_000_000, 60_000_000))\n        for k in range(steps):\n            d = date.today() - timedelta(days=k * 3)\n            dpi = min(0.78, max(0.30, base_dpi + rng.normal(0, 0.02)))\n            otc_total = int(base_vol * rng.uniform(0.7, 1.3))\n            otc_short = int(otc_total * dpi)\n            rows.append({"Ticker": ticker, "Date": d, "OTC_Short": otc_short,\n                         "OTC_Total": otc_total, "DPI": round(dpi, 6)})\n    return pd.DataFrame(rows)\n')
print("wrote off-exchange signal and ingestion updates")

wrote off-exchange signal and ingestion updates


In [4]:
!pip install -q -e .

import os, sys, importlib
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import quant_research; importlib.reload(quant_research)
print("Package ready:", quant_research.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for quant-equity-research (pyproject.toml) ... done
Package ready: 0.1.0


## Construct the signal

We compute as of today over a $60$-day lookback, treating the most recent $20$ days as
the window whose off-exchange share is compared against the prior level. Off-exchange
data is daily, so the windows are shorter than the event-driven signals.


In [5]:
from quant_research.ingestion.client import QuiverClient
from quant_research.signals.offexchange import OffExchangeSignal
import pandas as pd

client = QuiverClient(token=QUIVER_TOKEN) if USE_LIVE else QuiverClient(mock=True)
signal = OffExchangeSignal(client, lookback_days=60, min_oe_volume=1_000_000)

scores = signal.compute()
print(f"{len(scores)} tickers scored | data: {'live' if USE_LIVE else 'mock'}")

disp = scores.head(12).copy()
disp["dpi_%"] = (disp["dpi"] * 100).round(1)
disp["vol_$M"] = (disp["volume"] / 1e6).round(0)
disp[["score", "dpi_%", "vol_$M"]]

871 tickers scored | data: live


,score,dpi_%,vol_$M
ticker,,,
GDC,100.0,51.5,420.0
PTCT,90.7,88.6,3.0
NEO,86.0,84.3,2.0
BEKE,85.9,84.3,1.0
ADTX,85.5,56.9,246.0
RRC,84.6,83.0,1.0
SPWR,84.4,82.8,2.0
LAZ,82.8,81.4,1.0
QGEN,81.5,80.1,1.0


### Reading the output

`dpi_%` is the off-exchange short ratio (the buying-pressure proxy), and `vol_$M` is the
off-exchange volume. The `*_n` columns below are the normalized features feeding the
score.


In [6]:
comp_cols = [c for c in scores.columns if c.endswith("_n")]
scores[["score"] + comp_cols].head(8)

,score,dpi_n,volume_n
ticker,,,
GDC,100.0,0.567,1.000
PTCT,90.7,1.000,0.005
NEO,86.0,0.950,0.002
BEKE,85.9,0.949,0.001
ADTX,85.5,0.629,0.585
RRC,84.6,0.935,0.001
SPWR,84.4,0.932,0.002
LAZ,82.8,0.915,0.001


### Top names, shaded by share trend

Shading by `dpi` separates names ranking high on footprint from those ranking high on
the buying-pressure proxy.


In [7]:
import plotly.express as px

top = scores.head(15).reset_index()
fig = px.bar(top, x="score", y="ticker", orientation="h",
             color="dpi_n", color_continuous_scale="Greys",
             labels={"dpi_n": "DPI (short ratio)"},
             title="Off-exchange signal — top 15 (shading = DPI)")
fig.update_layout(yaxis=dict(autorange="reversed"), height=460)
fig.show()

## Reading it honestly

- **The DPI reading is contested.** Treating a high off-exchange short ratio as net
  buying rests on the market-maker-fill argument, which is a reasonable but debated
  interpretation. The signal leans on it lightly and is weighted lowest for this reason.
- **It is a snapshot, not history.** With one date per ticker there is no momentum or
  trend, and the live feed cannot be backtested historically — this is a current-state
  overlay that confirms what the event signals surface.
- **Liquidity matters.** The market-wide universe is full of illiquid names with
  meaningless ratios; the volume floor is what keeps the ranking on real names. Raise
  `min_oe_volume` to focus on the most heavily-traded names.

This completes the four-signal set. Because off-exchange has no history to backtest, the
composite work will lean on the three event signals for the historical test and treat
off-exchange as a present-day confirming layer.


## Commit

In [8]:
!git config --global user.email "aballa1234@gmail.com"
!git config --global user.name "Alex Balla"

!git add -A
!git commit -m "Add off-exchange signal"
!git push

[main 2c5bcca] Add off-exchange signal
 3 files changed, 122 insertions(+)
 create mode 100644 src/quant_research/signals/offexchange.py
Enumerating objects: 16, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (9/9), 3.02 KiB | 3.02 MiB/s, done.
Total 9 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/balla-a12/quant-equity-research.git
   86b77c1..2c5bcca  main -> main


## Recap and next

All four Hobbyist signals are now built on the shared contract: congress, government
contracts, lobbying, and off-exchange. The next notebook is the payoff — a composite
scorer that blends the four into one ranking (congress $40$, contracts $30$, lobbying
$20$, off-exchange $10$), followed by a backtest of the composite through the `02`
engine to see whether blending lifts the edge above any single signal. The dashboard
then surfaces today's top composite candidates with the historical evidence behind each.
